<a href="https://colab.research.google.com/github/RytisBalt/Ma-ininis-mokymasis/blob/KetvirtasKontrolinis/TMMA2_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

path = "Date_Fruit_Datasets.xlsx"
df = pd.read_excel(io=path,sheet_name="Date_Fruit_Datasets")

X, y = df.drop(['Class'],axis=1), df['Class']

# duomenys antrai užduočiai
X_train, X_test, y_train, y_test = train_test_split(X,y, train_size=0.75,stratify=y, random_state=0)
X.shape,y.shape

# duomenys pirmai užduočiai
X1, X_train1, X_test1, y1, y_train1, y_test1 = X.iloc[:,1:],X_train.iloc[:,1:],X_test.iloc[:,1:],X.iloc[:,0],X_train.iloc[:,0], X_test.iloc[:,0]




In [ ]:
#1 a)

from sklearn.feature_selection import mutual_info_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_percentage_error


mutual_info = mutual_info_regression(X_train1, y_train1)

mutual_info_df = pd.DataFrame({
    'Požymis': X_train1.columns,
    'Mutual_Information': mutual_info
}).sort_values(by='Mutual_Information', ascending=False)


top_features = mutual_info_df['Požymis'].head(15).values


print("Top 15 kintamųjų:")
print(top_features)

X_train_top = X_train1[top_features]
X_test_top = X_test1[top_features]

rfr = RandomForestRegressor()
rfr.fit(X_train_top, y_train1)

y_pred = rfr.predict(X_test_top)

print("\nAbipusės informacijos lentelė (Top 15):")
print(mutual_info_df.head(15))

r2_mir = r2_score(y_test1, y_pred)
mape_mir = mean_absolute_percentage_error(y_test1, y_pred)

# Print metrics
print(f"\nModelio vertinimo metrikos:")
print(f"R^2: {r2_mir:.4f}")
print(f"MAPE: {mape_mir:.4f}")

Top 15 kintamųjų:
['EQDIASQ' 'CONVEX_AREA' 'PERIMETER' 'MINOR_AXIS' 'MAJOR_AXIS'
 'SHAPEFACTOR_1' 'SHAPEFACTOR_2' 'EntropyRR' 'ECCENTRICITY' 'COMPACTNESS'
 'SHAPEFACTOR_3' 'ROUNDNESS' 'EntropyRG' 'EntropyRB' 'ALLdaub4RG']

Abipusės informacijos lentelė (Top 15):
          Požymis  Mutual_Information
4         EQDIASQ            5.095102
6     CONVEX_AREA            3.515601
0       PERIMETER            1.837679
2      MINOR_AXIS            1.424844
1      MAJOR_AXIS            1.199323
11  SHAPEFACTOR_1            1.145336
12  SHAPEFACTOR_2            1.107456
27      EntropyRR            0.406583
3    ECCENTRICITY            0.358372
10    COMPACTNESS            0.346069
13  SHAPEFACTOR_3            0.345055
9       ROUNDNESS            0.320043
28      EntropyRG            0.319888
29      EntropyRB            0.286870
31     ALLdaub4RG            0.273618

Modelio vertinimo metrikos:
R^2: 0.9997
MAPE: 0.0041


In [ ]:
#1 b)
from sklearn.feature_selection import SelectKBest

selector = SelectKBest(score_func=mutual_info_regression, k=15)
X_train_filtered_top = selector.fit_transform(X_train1, y_train1)
X_test_filtered_top = selector.transform(X_test1)

selected_features = X_train1.columns[selector.get_support()]
print("Top 15 kintamieji pagal abipusės informacijos filtrą:")
print(selected_features.values)

rfr = RandomForestRegressor()
rfr.fit(X_train_filtered_top, y_train1)

y_pred = rfr.predict(X_test_filtered_top)


r2_fil = r2_score(y_test1, y_pred)
mape_fil = mean_absolute_percentage_error(y_test1, y_pred)

print("\nModelio vertinimo metrikos:")
print(f"R^2: {r2_fil:.4f}")
print(f"MAPE: {mape_fil:.4f}")

mutual_info_scores = selector.scores_
mutual_info_df = pd.DataFrame({
    'Požymis': X_train1.columns,
    'Mutual_Information': mutual_info_scores
}).sort_values(by='Mutual_Information', ascending=False)

print("\nAbipusės informacijos lentelė (Top 15):")
print(mutual_info_df.head(15))


Top 15 kintamieji pagal abipusės informacijos filtrą:
['PERIMETER' 'MAJOR_AXIS' 'MINOR_AXIS' 'ECCENTRICITY' 'EQDIASQ'
 'CONVEX_AREA' 'ROUNDNESS' 'COMPACTNESS' 'SHAPEFACTOR_1' 'SHAPEFACTOR_2'
 'SHAPEFACTOR_3' 'EntropyRR' 'EntropyRG' 'EntropyRB' 'ALLdaub4RG']

Modelio vertinimo metrikos:
R^2: 0.9998
MAPE: 0.0039

Abipusės informacijos lentelė (Top 15):
          Požymis  Mutual_Information
4         EQDIASQ            5.095102
6     CONVEX_AREA            3.515899
0       PERIMETER            1.837624
2      MINOR_AXIS            1.424844
1      MAJOR_AXIS            1.199323
11  SHAPEFACTOR_1            1.148414
12  SHAPEFACTOR_2            1.106995
27      EntropyRR            0.406593
3    ECCENTRICITY            0.358732
10    COMPACTNESS            0.345907
13  SHAPEFACTOR_3            0.344528
9       ROUNDNESS            0.320208
28      EntropyRG            0.319888
29      EntropyRB            0.286905
31     ALLdaub4RG            0.273646


In [ ]:
#1 c)
from sklearn.linear_model import Lasso


lasso = Lasso(alpha=0.01)

lasso.fit(X_train1, y_train1)

lasso_coefficients = pd.DataFrame({
    'Požymis': X_train1.columns,
    'Koeficientas': np.abs(lasso.coef_)
}).sort_values(by='Koeficientas', ascending=False)

top_lasso_features = lasso_coefficients['Požymis'].head(15).values

print("Top 15 požymių pagal Lasso regresija:")
print(top_lasso_features)

X_train_top = X_train1[top_lasso_features]
X_test_top = X_test1[top_lasso_features]

rfr = RandomForestRegressor()
rfr.fit(X_train_top, y_train1)

y_pred = rfr.predict(X_test_top)

print(lasso_coefficients.head(15))

r2_las = r2_score(y_test1, y_pred)
mape_las = mean_absolute_percentage_error(y_test1, y_pred)

print(f"\nModelio vertinimo metrikos:")
print(f"R^2: {r2_las:.4f}")
print(f"MAPE: {mape_las:.4f}")


/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.509e+09, tolerance: 7.750e+08
  model = cd_fast.enet_coordinate_descent(


Top 15 požymių pagal Lasso regresija:
['SHAPEFACTOR_2' 'COMPACTNESS' 'SHAPEFACTOR_4' 'SOLIDITY' 'ROUNDNESS'
 'SHAPEFACTOR_1' 'SHAPEFACTOR_3' 'ECCENTRICITY' 'EXTENT' 'SkewRR'
 'MINOR_AXIS' 'SkewRG' 'SkewRB' 'EQDIASQ' 'KurtosisRR']
          Požymis  Koeficientas
12  SHAPEFACTOR_2  4.266785e+07
10    COMPACTNESS  3.195633e+05
14  SHAPEFACTOR_4  2.977813e+05
5        SOLIDITY  1.575704e+05
9       ROUNDNESS  9.029674e+04
11  SHAPEFACTOR_1  6.296458e+04
13  SHAPEFACTOR_3  3.394705e+04
3    ECCENTRICITY  2.050074e+04
7          EXTENT  1.615199e+03
21         SkewRR  6.643908e+02
2      MINOR_AXIS  5.248086e+02
22         SkewRG  4.234429e+02
23         SkewRB  2.832796e+02
4         EQDIASQ  1.139295e+02
24     KurtosisRR  9.969779e+01

Modelio vertinimo metrikos:
R^2: 0.9997
MAPE: 0.0042


Gauname palyginti šitų dalių modelius pagal metrikas, mums reiktų artimesnės 1 R^2 reikšmės ir kuo mažesnės paklaidos. Šis atvejis yra abipusės informacijos kiekio su K-best filtru. Kur R^2 = 0,9998 ir MAPE = 0,0039



In [ ]:
#2 a)
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

scaler = MinMaxScaler(feature_range=(-1, 1))


pca = PCA()
param_grid = {
    'pca__n_components': [2, 4, 6, 8, 16, 32],
    'logistic__C': [0.01, 0.1, 1, 10, 100]
}
logistic_model = LogisticRegression(max_iter=10000)

pipeline = Pipeline(steps=[
    ('scaler', scaler),
    ('pca', pca),
    ('logistic', logistic_model)
])

grid_search = GridSearchCV(estimator=pipeline, param_grid=param_grid, cv=3, scoring='accuracy')

grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("Geriausias reguliarizacijos parametras:", grid_search.best_params_['logistic__C'])
print("Best number of PCA components:", grid_search.best_params_['pca__n_components'])
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Geriausias reguliarizacijos parametras: 10
Best number of PCA components: 16

Classification Report:
              precision    recall  f1-score   support

       BERHI       0.81      0.81      0.81        16
      DEGLET       0.81      0.84      0.82        25
       DOKOL       0.96      0.92      0.94        51
       IRAQI       0.94      0.83      0.88        18
      ROTANA       0.98      1.00      0.99        42
      SAFAVI       0.98      1.00      0.99        50
       SOGAY       0.79      0.83      0.81        23

    accuracy                           0.92       225
   macro avg       0.90      0.89      0.89       225
weighted avg       0.92      0.92      0.92       225



In [ ]:
#2 b)
from sklearn.decomposition import  KernelPCA
from sklearn.preprocessing import StandardScaler


scaler = StandardScaler()

logistic_model = LogisticRegression(penalty='l1', solver = 'liblinear')

pipeline = Pipeline(steps=[
    ('scaler', scaler),
    ('decomposer', PCA()),
    ('logistic', logistic_model)
])

param_grid = [
    {
        'decomposer': [PCA(n_components=15)],
        'logistic__C': [0.01, 0.1, 1, 10, 100]
    },
    {
        'decomposer': [KernelPCA(kernel="rbf", n_components=15)],
        'decomposer__gamma': [0.1, 1, 10],
        'logistic__C': [0.01, 0.1, 1, 10, 100]
    }
]


grid_search = GridSearchCV(estimator=pipeline, param_grid=param_grid, cv=3, scoring='accuracy')

grid_search.fit(X_train, y_train)

y_pred = grid_search.best_estimator_.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("Best Decomposer:", grid_search.best_params_['decomposer'])
print("Best Logistic Regression C:", grid_search.best_params_['logistic__C'])
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

pca_model = grid_search.best_estimator_.named_steps['decomposer']
feature_importances = np.abs(pca_model.components_).sum(axis=0)
feature_importances_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': feature_importances
}).sort_values(by='Importance', ascending=False)

print("\nSignificant Features (PCA):")
print(feature_importances_df.head(15))

/usr/local/lib/python3.10/dist-packages/sklearn/svm/_base.py:1243: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/svm/_base.py:1243: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/svm/_base.py:1243: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Best Decomposer: PCA(n_components=15)
Best Logistic Regression C: 1

Classification Report:
              precision    recall  f1-score   support

       BERHI       0.80      1.00      0.89        16
      DEGLET       0.80      0.64      0.71        25
       DOKOL       0.94      0.92      0.93        51
       IRAQI       1.00      0.83      0.91        18
      ROTANA       0.87      0.98      0.92        42
      SAFAVI       0.96      1.00      0.98        50
       SOGAY       0.81      0.74      0.77        23

    accuracy                           0.90       225
   macro avg       0.88      0.87      0.87       225
weighted avg       0.90      0.90      0.89       225


Significant Features (PCA):
          Feature  Importance
27     KurtosisRB    3.271729
24         SkewRB    2.921835
26     KurtosisRG    2.705882
21       StdDevRB    2.577962
8          EXTENT    2.562167
25     KurtosisRR    2.561816
20       StdDevRG    2.488318
15  SHAPEFACTOR_4    2.351666
30      Entr

In [ ]:
#2 c)
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.metrics import classification_report

scaler = StandardScaler()

feature_selector = SelectKBest(score_func=mutual_info_classif, k=15)

pca = PCA(n_components=15)
kernel_pca = KernelPCA(kernel="rbf", n_components=15)

logistic_model = LogisticRegression(penalty='l2', solver='lbfgs', max_iter=10000)


pipeline = Pipeline(steps=[
    ('scaler', scaler),
    ('feature_selector', feature_selector),
    ('decomposer', PCA()),
    ('logistic', logistic_model)
])


param_grid = [
    {
        'decomposer': [PCA()],
        'logistic__C': [0.1, 1, 10]
    },
    {
        'decomposer': [KernelPCA(kernel="rbf")],
        'decomposer__gamma': [0.1, 1, 10],
        'logistic__C': [0.1, 1, 10]
    }
]

grid_search = GridSearchCV(estimator=pipeline, param_grid=param_grid, cv=3, scoring='accuracy')

grid_search.fit(X_train, y_train)

y_pred = grid_search.best_estimator_.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("Best Decomposer:", grid_search.best_params_['decomposer'])
print("Best Logistic Regression C:", grid_search.best_params_['logistic__C'])
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
pca_model = grid_search.best_estimator_.named_steps['decomposer']

feature_importances = np.abs(pca_model.components_).sum(axis=0)
feature_importances_df = pd.DataFrame({
        'Feature': selected_features,
        'Importance': feature_importances
    }).sort_values(by='Importance', ascending=False)

print("\nSignificant Features (PCA):")
print(feature_importances_df.head(15))


Best Decomposer: PCA()
Best Logistic Regression C: 10

Classification Report:
              precision    recall  f1-score   support

       BERHI       0.73      0.50      0.59        16
      DEGLET       0.71      0.68      0.69        25
       DOKOL       0.94      0.92      0.93        51
       IRAQI       0.79      0.83      0.81        18
      ROTANA       0.88      1.00      0.93        42
      SAFAVI       0.98      0.98      0.98        50
       SOGAY       0.74      0.74      0.74        23

    accuracy                           0.87       225
   macro avg       0.82      0.81      0.81       225
weighted avg       0.86      0.87      0.86       225


Significant Features (PCA):
          Feature  Importance
3    ECCENTRICITY    2.979745
2      MINOR_AXIS    2.962387
0       PERIMETER    2.884434
5     CONVEX_AREA    2.840361
9   SHAPEFACTOR_2    2.787247
14     ALLdaub4RG    2.786797
8   SHAPEFACTOR_1    2.658758
13      EntropyRB    2.658178
7     COMPACTNESS    2.562

Gavome tris klasifikavimo lenteles, pagal mane žiūrėsim į geriausia macro ir weighted avg kombinaciją, šiuo atveju tai yra pats pirmas modelis su 0.90 ir 0.92, bet kiti nežymiai skiriasi, naudočiau pirmąjį modelį. Atrenkami b variante pagal importance, c variante pritaikome filtrą.